# ECON 398 Project Workbook

Authors: *William Clinton Co, Jerrold Huang, Jae Park, Kaiyan Zhang, Irene Berezin*

In [1]:
library(tidyverse)
library(ggplot2)
library(haven)
library(lubridate)
library(stargazer)
library(units)
library(broom)
library(sf)

Warning message:
"package 'tidyverse' was built under R version 4.4.3"
Warning message:
"package 'ggplot2' was built under R version 4.4.2"


Warning message:
"package 'dplyr' was built under R version 4.4.2"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Please cite as: 


 Hlavac, Marek (2022). stargazer: Well-Formatted Regression and Summary Statistics Tables.

 R package version 5.2.3. https://CRAN.R-project.org/package=stargazer 


udunits database from C:/Users/Kaiyan Zhang/AppData/Local/R/win-library/4.4/units/share/udunits/udunits2.xml

Linking to GEOS 3.12.1, GDAL 3.8.4, PROJ 9.3.1; sf_use_s2() is TRUE



In [9]:
# reading in cancensus data for 2021
library(cancensus)
options(cancensus.api_key='CensusMapper_85cff6944226682c2124e67a6d1cc29c')
options(cancensus.cache_path = "/census_api")

census2021 <- get_census(dataset='CA21', 
regions=list(CSD="5915022"), vectors=c("v_CA21_1","v_CA21_6","v_CA21_386","v_CA21_906","v_CA21_5901", "v_CA21_4410"), 
labels="detailed", 
geo_format="sf", 
level='DA',
use_cache = FALSE)

census2021_clean <- census2021|>
    rename(
        geoid = `GeoUID`,
        households = Households, 
        dwellings = Dwellings, 
        population =  `v_CA21_1: Population, 2021`,
        population_density = `v_CA21_6: Population density per square kilometre`, 
        region = `Region Name`, 
        area_sq_km = `Area (sq km)`, 
        age =  `v_CA21_386: Average age`, 
        income = `v_CA21_906: Median total income of household in 2020 ($)`,
        education = `v_CA21_5901: University certificate or diploma above bachelor level`,
        immigrant = `v_CA21_4410: Immigrants`    
    )|>
    filter(region == "Vancouver")|>
    mutate(immigrant_prop = immigrant/population) |>
    mutate(education_prop = education / population)|>
    mutate(income = income / 100000)|>
    select(name, households, dwellings, population,population_density, region, area_sq_km, age, income, immigrant_prop, education_prop)|>
    glimpse()

census2021_st <- st_sf(census2021_clean, geometry = census2021_clean$geometry, crs = 4326)
census2021_centroid <- st_centroid(census2021_clean)

Querying CensusMapper API...



Downloading: 26 kB     

Querying CensusMapper API...



Downloading: 260 kB     Rows: 0
Columns: 12
$ name               <chr> 
$ households         <int> 
$ dwellings          <int> 
$ population         <dbl> 
$ population_density <dbl> 
$ region             <fct> 
$ area_sq_km         <dbl> 
$ age                <dbl> 
$ income             <dbl> 
$ immigrant_prop     <dbl> 
$ education_prop     <dbl> 
$ geometry           <GEOMETRY [°]> 
